# RankAdaptation — α Sweep on L4 GPU
Tests L8 steering at multiple α values on Qwen2.5-7B-Instruct.

In [ ]:
# Install deps
!pip install -q transformers datasets bitsandbytes accelerate
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

# Clone repo
!git clone https://github.com/anomalyco/RankAdaptation.git
%cd RankAdaptation

In [ ]:
# Remove old .pt checkpoints from the repo (large files not tracked)
# The TT will be trained locally
!rm -f best_*.pt

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Train standard TT on GSM8K test generations (data generated on-the-fly)
# First collect a small set of trajectories from GSM8K test
import torch, sys, os, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
sys.path.insert(0, '.')
from src.adapters.trajectory_transformer import TrajectoryTransformer

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print('Loading model (4-bit)...')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True,
                                              quantization_config=quant, device_map='auto')
model.eval()
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tok.pad_token = tok.eos_token
print('Loaded')

In [ ]:
# Quick TT training on generated data
# We create a small trajectory set for the TT
import numpy as np

device = 'cuda'
N_LAYERS = 28
D_MODEL = 3584

ds = load_dataset('openai/gsm8k', 'main', split='test')
examples = ('Q: Janet has 5 ducks. She buys 3 more. How many?\nA: Step 1: 5. Step 2: buy 3. Step 3: 5+3=8. So answer is 8.\n\n'
            'Q: 12 muffins/hr, 8 hours?\nA: Step 1: 12/hr. Step 2: 8h. Step 3: 12x8=96. So answer is 96.\n\n')

trajs = []
for i in range(200):
    prob = ds[i]
    prompt = f"{examples}Q: {prob['question']}\nA:"
    inp = tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(inp.input_ids, use_cache=True, output_hidden_states=True)
    hs = out.hidden_states
    for pos in range(inp.input_ids.shape[1] - 1):
        h_at = torch.stack([hs[li][0, pos].float() for li in range(N_LAYERS)], dim=0)
        h_next = torch.stack([hs[li][0, pos + 1].float() for li in range(N_LAYERS)], dim=0)
        trajs.append((h_at, h_next - h_at))
    if (i+1) % 50 == 0:
        print(f'  [{i+1}/200] {len(trajs)} trajs')

h = torch.stack([t[0] for t in trajs])
v = torch.stack([t[1] for t in trajs])
print(f'Collected {len(trajs)} trajectories')

# Split
perm = torch.randperm(len(h))
n_val = len(h) // 10
tr_h, tr_v = h[perm[n_val:]].to(device), v[perm[n_val:]].to(device)
va_h, va_v = h[perm[:n_val]].to(device), v[perm[:n_val]].to(device)

In [ ]:
# Train TT
v_mean = tr_v.mean(dim=(0, 1), keepdim=True)
v_std = tr_v.std(dim=(0, 1), keepdim=True) + 1e-8
tr_vn = (tr_v - v_mean) / v_std
va_vn = (va_v - v_mean) / v_std

tt = TrajectoryTransformer(d_model=768, n_layers=6, n_heads=8, d_ff=3072,
                            n_positions=N_LAYERS, d_input=D_MODEL).to(device)
opt = torch.optim.AdamW(tt.parameters(), lr=3e-4)
bs = 64

for ep in range(15):
    tt.train()
    perm = torch.randperm(len(tr_h))
    for i in range(0, len(tr_h), bs):
        idx = perm[i:i+bs]
        vp = tt(tr_h[idx])
        loss = torch.nn.functional.mse_loss(vp, tr_vn[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    
    tt.eval()
    with torch.no_grad():
        vp = tt(va_h)
        r2 = 1 - (vp - va_vn).pow(2).mean() / va_vn.var().item()
    print(f'ep={ep+1:2d} val_r2={r2:.4f}')

torch.save(tt.state_dict(), 'best_gen_tt_7b.pt')
print('TT saved')

In [ ]:
!python run_alpha_sweep.py --n-test 100 --layer 8 --alphas -0.5 -0.2 -0.1 0.0 0.05 0.1 0.2 0.5 1.0 2>&1

In [ ]:
# Show results
import json
results = json.load(open('alpha_sweep_L8.json'))
baseline = results['baseline']
print(f"{'Alpha':>6} {'Acc':>8} {'Delta':>8}")
print(f"{'-----':>6} {'---':>8} {'-----':>8}")
for a_str, acc in sorted(results['alphas'].items()):
    a = float(a_str)
    d = acc - baseline if a != 0.0 else 0
    print(f"{a:>6.2f} {100*acc:>7.1f}% {100*d:>+7.1f}pp")

# Download results
from google.colab import files
files.download('alpha_sweep_L8.json')